# 95. 웹 사이트 로딩 속도 및 성능 분석 실습

> **📥 [실습 주피터 노트북(.ipynb) 다운로드](practice.ipynb)**
## 실전 데이터 분석 95: 글로벌 웹 서비스의 서버 지역 리전 및 CDN 캐시 유무별 페이지 용량 대비 로딩 속도(p95) 분석

![도입 만화](img/intro_comic.png)

### 📊 실습 개요 (Tutorial Overview)
글로벌 웹 애플리케이션의 웹 트래픽 리소스로더 로그 데이터셋입니다. 페이지 전송 용량(PageSize_MB), 동시 접속 요청 수(RequestCount), CDN 캐시 히트 상태(CacheStatus)가 최종 사용자의 모바일 렌더링 로딩 지연 속도(LoadTime_ms)에 미치는 병목 유발 인자를 분석하고 시각화합니다.

---

### 🛠️ 핵심 분석 실습 역량 (Core Skills)
* **이상치 억제 중앙값 대치 (\`fillna\`):** 순간 트래픽 수집 센서 누락 건을 극단값 왜곡이 적은 동시 접속 중앙값(Median)으로 대치합니다.
* **캐시 유무별 속도 대조 및 리소스 비례 분석 (\`boxplot\` , \`scatterplot\`):** CDN 캐시 여부별 로딩 편차 상자 요약 및 페이지 용량 대비 로딩 속도의 캐시 오버레이 상관 산점도를 시각화합니다.

---

## 🧐 실무 도메인 지식 가이드 (Domain Knowledge Guide)

> **👥 인사 및 조직 문화 (HR & Workplace Analysis)**
> 인적 자원 관리(HR Analytics)는 핵심 인재의 이탈(Attrition) 방지, 사내 직무 이동 성과, 복지 리텐션을 다루는 과학적 기업 운영 분야입니다.
>
> * **직원 자발적 퇴사(Attrition)**: 야근 빈도, 직무 몰입도(Engagement), 급여 대비 승진 연한 격차 등을 통해 조기 이탈 리스크 직원을 경보합니다.
> * **채용 채널 성과 분석**: 직무 코딩테스트 및 전형 결과와 입사 사후 실제 성과 데이터 간의 타당성(상관) 관계를 실증 분석합니다.
> * **원격 근무 효율성**: 원격/사무실 하이브리드 근무자의 생산성 점수와 근무 만족도 분산 차이를 통계 검증(T-test 등)합니다.

---

## Step 1: 데이터 불러오기 및 기본 정보 확인 (Data Load)

![Step 1 데이터 수집 개념도](img/step1_dataset.svg)

제공된 CSV 파일을 분석 컴퓨터로 불러와 로드하고, 컬럼의 타입 및 데이터 구조를 확인합니다.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트 설정
import koreanize_matplotlib
sns.set_theme(style="whitegrid")

# 데이터 로드
df = pd.read_csv('./website_performance.csv')
print(df.info())
print(df.head())


RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   RequestID     1000 non-null   int64  
 1   ServerRegion  1000 non-null   str    
 2   PageSize_MB   1000 non-null   float64
 3   RequestCount  985 non-null    float64
 4   CacheStatus   1000 non-null   str    
 5   LoadTime_ms   1000 non-null   float64
dtypes: float64(3), int64(1), str(2)
memory usage: 47.0 KB
> 
> RequestID ServerRegion  PageSize_MB  RequestCount CacheStatus  LoadTime_ms
0     950001      AP-East         6.03         247.0         Hit       1264.2
1     950002      US-East         7.74         417.0         Hit       1573.7
2     950003      US-East         3.69           NaN         Hit        801.3
3     950004      AP-East         6.54         305.0         Hit       1494.6
4     950005      AP-East         0.95         160.0         Hit        317.8
> ```

---

## Step 2: 결측치 및 데이터 정제 (Data Cleaning)

![Step 2 데이터 정제 개념도](img/step2_missing_values.svg)

수집 과정에서 빈칸으로 기록된 결측값(NaN)의 존재 여부를 진단하고, 데이터 도메인 성격에 부합하는 통계 수치 대입 기법으로 정제합니다.


In [ ]:
# 1. 컬럼별 결측치 개수 확인
print("--- 정제 전 결측치 확인 ---")
print(df.isnull().sum())

# 2. 결측치 전처리 및 대치 수행
median_req = df['RequestCount'].median()
df['RequestCount'] = df['RequestCount'].fillna(median_req)
print(df.isnull().sum())


ServerRegion     0
PageSize_MB      0
RequestCount    15
CacheStatus      0
LoadTime_ms      0
> 
> --- 정제 후 결측치 확인 ---
> RequestID       0
ServerRegion    0
PageSize_MB     0
RequestCount    0
CacheStatus     0
LoadTime_ms     0
> ```

### 💡 분석가의 통찰 (Analyst's Insight)
* **접속량 지표 중앙값 대입의 트래픽 분석 타당성:** 동시 요청 접속자 수는 디도스(DDoS) 공격이나 파격 세일 시점에 비상식적인 수만 명 단위 폭발로 롱테일 우측 왜도가 극단적으로 형성되는 피크 통계 지표입니다. 단순 평균값으로 결측을 채우면 일상의 평범한 트래픽 로그가 수천 명 수준으로 강제 상향되어 망 리소스 설계를 그르칩니다. 중앙값 대치가 타당합니다.

---

## Step 3: 단변수 분포 분석 (Univariate EDA)

![Step 3 시각화 개념도](img/step3_visualization.svg)

가장 먼저 핵심 변수가 전체 데이터에서 어떤 빈도와 분포를 가졌는지 단일 변수 시각화를 통해 파악해 봅니다.


In [ ]:
plt.figure(figsize=(8, 5))

# 단변수 분석 그래프 생성
sns.boxplot(data=df, x='CacheStatus', y='LoadTime_ms', palette='coolwarm')
plt.title('CDN 캐시 상태(Cache Status)별 웹 페이지 로딩 속도 비교', fontsize=14, fontweight='bold')
plt.show()


### 💡 시각화 차트 읽는 법 & 인사이트
* **캐시 히트가 보장하는 쾌속 로딩 속도 증명:** 캐시 히트(Hit) 그룹의 로딩 속도 상자 범위는 캐시 미스(Miss) 그룹 상자의 1/3 높이 바닥에 극도로 작게 수축해 포진해 있습니다. 이는 CDN 도입 인프라 투자가 사용자 체감 반응 속도 격차 개선에 절대적 기여를 함을 정량적 상자 밴드로 입증한 것입니다.

---

## Step 4: 다변수 상관관계 및 이상치 분석 (Multivariate EDA)

![Step 4 상관관계 분석 개념도](img/step4_multivariate.svg)

두 개 이상의 변수를 동시에 결합하여, 조건에 따른 수치 차이나 독립 변수와 종속 변수 간의 통계적 경향을 분석합니다.


In [ ]:
plt.figure(figsize=(9, 6))

# 다변수 분석 그래프 생성
sns.scatterplot(data=df, x='PageSize_MB', y='LoadTime_ms', hue='CacheStatus', palette='Set1', alpha=0.8)
plt.title('전송 용량 대비 페이지 로딩 속도와 캐시 연동 상관성', fontsize=14, fontweight='bold')
plt.show()


### 💡 코드 딥다이브 & 비즈니스 통찰 (Analyst's Insight)
* **용량 증가에 직결되는 지연 지스 곡선 및 캐시 무력화 방어 효과:** 페이지 리소스 용량(X축)이 증가할수록 로딩 속도(Y축)가 가파르게 우상향합니다. 특히 주목할 점은 캐시 미스(Miss, 빨간색 점) 그룹은 용량 증가에 따라 경사도가 매우 가파르게 상승하여 대역폭 한계 지연을 겪는 반면, 캐시 히트(Hit, 파란색 점) 그룹은 용량이 늘어나더라도 상대적으로 매우 평평한 수평 지연 띠를 유지해 캐싱 기술의 부하 방어 기여를 시각적으로 선명히 증명합니다.

---

## Step 5: 통계적 직관과 해석 (Statistical Logic)

> 💡 **[웹 성능 최적화와 가장 큰 콘텐츠 렌더링 시간(LCP)의 통계 개선]**
... (생략: 웹 성능 코어 웹 바이탈 LCP 개선 및 캐싱 효과 수학적 요약)

---

## 🎯 30분 강의 마무리 및 심화 과제

오늘 우리는 실전 데이터셋을 분석하여 판다스로 데이터를 가공 및 정제하고, 시각화를 활용하여 핵심 변수 간의 통계적 유의성을 검증했습니다. 데이터 속에서 숨겨진 패턴을 올바른 시각으로 탐색하는 능력이 데이터 사이언티스트의 가장 강력한 무기입니다.

### 📝 심화 과제 (Advanced Challenge)
* **서버 리전(ServerRegion)별 평균 로딩 속도 비교:** 아시아, 유럽, 미국 권역별 평균 \`LoadTime_ms\`를 요약 테이블로 도출하고 지리적 정체 격차를 파악해 보세요.
* **성능 불량 임계 페이지 탐색:** 로딩 속도가 3초(3000ms) 이상 소요된 트래픽 건들 중 캐시 상태가 Miss였던 비율을 산출해 CDN 캐시 튜닝 필요 비율을 구해보세요.
